In [15]:
import kagglehub
import os
import nltk
from nltk.corpus import stopwords
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

### 1. Загрузка данных

In [2]:
path = kagglehub.dataset_download("ankurzing/sentiment-analysis-for-financial-news")
files = os.listdir(path)
print(files)

['all-data.csv', 'FinancialPhraseBank']


In [3]:
file_path = os.path.join(path, "all-data.csv")
df = pd.read_csv(file_path, header=None)
df = df.rename(columns={0: 'class', 1: 'text'})
df[:5]

,class,text
0,neutral,"According to Gran , the company has no plans t..."
1,neutral,Technopolis plans to develop in stages an area...
2,negative,The international electronic industry company ...
3,positive,With the new production plant the company woul...
4,positive,According to the company 's updated strategy f...


### 2.Подготовка данных

#### 2.1 Посчитаем tf-idf и удалим редко встречающиеся слова

Исходя из анализа в EDA.ipynb, было принято решение оставить стоп-слова, потому что многие из них имеют эмоциональную окраску (отсутствие отрицания или слова с отрицанием). Однако стоит удалить очень редкие слова, которые, вероятно, будут названиями компаний или деятельности компании. Данная информация не влияет на эмоциональный окрас и просто может создавать лишний шум.

Для сравнения результата можем обучить логистическую регрессию на двух выборках: на выборке со стоп-словами и на выборке без стоп-слов и сравнить результат.

In [11]:
import re

def remove_punctuation(text):
    return re.sub(r'[^a-zA-Z\s]', ' ', text)

def prepare_text(df):
    df = df.copy()
    df["text"] = df["text"].apply(lambda x: remove_punctuation(x))
    df["text"] = df["text"].apply(lambda x: x.lower())
    return df

In [12]:
df = prepare_text(df)

In [34]:
X_train, X_test, y_train, y_test = train_test_split(df["text"], df["class"], test_size=0.3, random_state=42)

In [46]:
stop_words = stopwords.words("english")
vec_full = TfidfVectorizer(min_df = 5)
vec_stop = TfidfVectorizer(stop_words=stop_words, min_df = 5)

X_train_full = vec_full.fit_transform(X_train)
X_test_full = vec_full.transform(X_test)
X_train_stop = vec_stop.fit_transform(X_train)
X_test_stop = vec_stop.transform(X_test)

### 3. Обучение базовой модели

#### 3.1 Обучение логистической регрессии на полной выборке

In [47]:
clf_full = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=1000,
    class_weight="balanced"
)

In [48]:
clf_full.fit(X_train_full, y_train)
y_pred_full = clf_full.predict(X_test_full)

In [49]:
print(classification_report(y_test, y_pred_full))

              precision    recall  f1-score   support

    negative       0.59      0.70      0.64       179
     neutral       0.83      0.80      0.82       847
    positive       0.69      0.68      0.69       428

    accuracy                           0.76      1454
   macro avg       0.70      0.73      0.72      1454
weighted avg       0.76      0.76      0.76      1454



#### 3.2 Обучение логистической регрессии на выборке без стоп-слов

In [59]:
clf_stop = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=1000,
    class_weight="balanced"
)

In [60]:
clf_stop.fit(X_train_stop, y_train)
y_pred_stop = clf_stop.predict(X_test_stop)

In [61]:
print(classification_report(y_test, y_pred_stop))

              precision    recall  f1-score   support

    negative       0.54      0.68      0.60       179
     neutral       0.81      0.78      0.79       847
    positive       0.65      0.62      0.63       428

    accuracy                           0.72      1454
   macro avg       0.66      0.69      0.68      1454
weighted avg       0.73      0.72      0.72      1454



### 4. Сравнение метрик

Линейные модели для данной задачи справляются неплохо, однако первая модель (где не были удалены самые распространенные слова) справляется чуть лучше: это видно и по accuracy, и по f1 метрике, что показывает более сбалансированную работу по всем классам.

Однако для задач, где требуется большая точность, такая модель имеет недостаточно хорошую предсказательную способность. Примером такой задачи может быть финансовый анализ акций: для этого может потребоваться проанализировать эмоциональную тональность новостей о компании, чтобы сделать вывод о целесообразности покупки акций. В данном случае модель достаточно плохо предсказывает негативный класс, что может быть критически важным в данной задаче.